# Wayne → XSOAR detections — query notebook

Validate the detections **poll** and raw-log **drill-back** before wiring the Cortex XSOAR
"AWS - Athena - Beta" integration. Each section below is one Athena query.

**Setup:** `pip install boto3`, keep `xsoar-detections.env` (the credential) next to this
notebook, then run the cells top to bottom.

In [27]:
import json, time, boto3

REGION    = "ap-southeast-1"
WORKGROUP = "wayne"
RESULTS   = "s3://mss-log-fabric-8162-athena-results/wayne/"
ROLE_ARN  = "arn:aws:iam::942510828162:role/detections-reader"

# Read the static credential from xsoar-detections.env and assume the reader role.
def load_env(path="xsoar-detections.env"):
    env = {}
    for line in open(path):
        line = line.strip()
        if line.startswith("export "):
            line = line[len("export "):]
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            env[k] = v
    return env

env   = load_env()
creds = boto3.Session(
    aws_access_key_id=env["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=env["AWS_SECRET_ACCESS_KEY"],
    region_name=REGION,
).client("sts").assume_role(RoleArn=ROLE_ARN, RoleSessionName="xsoar-detections")["Credentials"]

athena = boto3.Session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=REGION,
).client("athena")
print("assumed", ROLE_ARN)

assumed arn:aws:iam::942510828162:role/detections-reader


In [28]:
def run_sql(sql):
    """Run an Athena query, wait, return rows as a list of dicts."""
    qid = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": "baselines", "Catalog": "AwsDataCatalog"},
        ResultConfiguration={"OutputLocation": RESULTS},
        WorkGroup=WORKGROUP,
    )["QueryExecutionId"]
    while True:
        s = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        if s["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1.5)
    if s["State"] != "SUCCEEDED":
        raise RuntimeError(s.get("StateChangeReason", s["State"]))
    rows, token = [], None
    while True:
        kw = {"QueryExecutionId": qid, "MaxResults": 1000}
        if token:
            kw["NextToken"] = token
        r = athena.get_query_results(**kw)
        rows += [[c.get("VarCharValue") for c in row["Data"]] for row in r["ResultSet"]["Rows"]]
        token = r.get("NextToken")
        if not token:
            break
    header = rows[0]
    return [dict(zip(header, row)) for row in rows[1:]]

## 1. Poll — detections newer than the watermark

Set `WATERMARK` to your starting point (this is XSOAR's "First fetch timestamp"). On later
runs set it to the largest `detected_at` you last saw — the cell prints that value for you.

In [30]:
WATERMARK = "2026-07-06 00:00:00"   # <-- your starting point
TENANT    = "suju"

poll_sql = f"""
SELECT dedup_key, rule_id, severity, status, identity, entity_type,
       event_time, evidence_count, window_start, window_end,
       event_details, source_event_id, detected_at
FROM   baselines.detections
WHERE  tenant_id = '{TENANT}'
  AND  status = 'ES'
  AND  detected_at > TIMESTAMP '{WATERMARK}'
  AND  event_time  > TIMESTAMP '{WATERMARK}' - INTERVAL '1' DAY
ORDER BY detected_at
"""

detections = run_sql(poll_sql)
print(len(detections), "detection(s)")
for d in detections:
    print(f"{d['detected_at']} {d['event_time']} [{d['severity']:<4}] {d['rule_id']:<30} {d['identity']:<30} {d['dedup_key']}")

if detections:
    print("\nnext WATERMARK =", max(d["detected_at"] for d in detections))

1 detection(s)
2026-07-06 01:31:16.923000 2026-07-06 01:00:00.000000 [2   ] suju-ftg-09-hscan-int-int      10.208.48.20                   4D56284448E77B946C81A947A9E4B2B154F61A33DF733134CF8C52D4378BA6CA

next WATERMARK = 2026-07-06 01:31:16.923000


## 2. Variables for one detection (`event_details`)

`event_details` is a JSON string of the analyst variables — `json.loads` it; all values are
strings. This is what populates the XSOAR incident fields.

In [31]:
DEDUP_KEY = detections[-1]["dedup_key"]    
# DEDUP_KEY = "7840BD0E9DE1DA31C0FB4A676EF44D2B4ACD08F39D3C2340875342B69112E2B9" # or paste a key from the poll output above

det    = next(d for d in detections if d["dedup_key"] == DEDUP_KEY)
labels = json.loads(det["event_details"])
labels

{'IBC_LogSourceName': '["SMAUBNEFWP001","SMAUSYDFWP001"]',
 'Vendor.action': '["accept","ip-conn","deny"]',
 'Vendor.subtype': '["forward","local"]',
 '_drillback': '{"from":"2026-07-06 01:00:00.000","like":["srcip=10.208.48.20 ","dstport=161 "],"table":"sources.suju__fortigate_hourly","to":"2026-07-06 01:05:00.000"}',
 'dc_dest_ip': '1010',
 'dc_event.action': '2',
 'destination.ip': '["10.100.45.105","10.100.46.175","10.100.45.159","10.100.46.25","10.100.45.13","10.100.47.247","10.100.44.83","10.100.46.53","10.100.47.121","10.100.44.64","10.100.47.78","10.100.44.45","10.100.45.92","10.100.44.140","10.100.44.185","10.100.44.55","10.100.45.43","10.100.47.17","10.100.47.120","10.100.45.76","10.100.47.37","10.100.46.244","10.100.46.65","10.100.47.193","10.100.47.80","10.100.44.44","10.100.47.38","10.100.44.254","10.100.45.174","10.100.46.217","10.100.47.137","10.100.45.98","10.100.47.8","10.100.44.77","10.100.46.197","10.100.44.91","10.100.44.122","10.100.47.4","10.100.44.184","10.100.44

## 3. Drill back to the raw FortiGate logs

Each detection carries a `_drillback` recipe inside `event_details`
(`{table, from, to, like[]}` — itself a JSON string). One query per detection; run it only
when an analyst opens the incident, since it scans GBs of raw logs.

In [32]:
db    = json.loads(labels["_drillback"])          # {table, from, to, like[]}
frm   = db["from"][:19]
to    = db["to"][:19]
likes = "".join(f"  AND message LIKE '%{p}%'\n" for p in db["like"])

# prune to the hour partitions the window falls in
parts = {(t[:4], t[5:7], t[8:10], t[11:13]) for t in (frm, to)}
prune = " OR ".join(
    f"(year='{y}' AND month='{m}' AND day='{d}' AND hour='{h}')"
    for y, m, d, h in sorted(parts))

drillback_sql = f"""
SELECT from_unixtime(TRY_CAST(_time AS double)) AS start_time, message AS rawstring
FROM   {db['table']}
WHERE  ({prune})
{likes}  AND TRY_CAST(regexp_extract(message, 'eventtime=([0-9]+)', 1) AS double) / 1e9
         BETWEEN to_unixtime(timestamp '{frm}') AND to_unixtime(timestamp '{to}')
ORDER BY start_time
LIMIT 1000
"""
print(drillback_sql)

raw = run_sql(drillback_sql)
print(len(raw), "raw line(s)\n")
for r in raw[:20]:
    print(r["start_time"], r["rawstring"])


SELECT from_unixtime(TRY_CAST(_time AS double)) AS start_time, message AS rawstring
FROM   sources.suju__fortigate_hourly
WHERE  ((year='2026' AND month='07' AND day='06' AND hour='01'))
  AND message LIKE '%srcip=10.208.48.20 %'
  AND message LIKE '%dstport=161 %'
  AND TRY_CAST(regexp_extract(message, 'eventtime=([0-9]+)', 1) AS double) / 1e9
         BETWEEN to_unixtime(timestamp '2026-07-06 01:00:00') AND to_unixtime(timestamp '2026-07-06 01:05:00')
ORDER BY start_time
LIMIT 1000

1000 raw line(s)

2026-07-06 01:00:47.302 logver=704112878 timestamp=1783299645 date=2026-07-06 time=11:00:45 eventtime=1783299645976804706 tz="+1000" logid="0001000014" type="traffic" subtype="local" level="notice" vd="root" srcip=10.208.48.20 srcport=60717 srcintf="port11" srcintfrole="undefined" dstip=10.100.44.1 dstport=161 dstintf="root" dstintfrole="undefined" srccountry="Reserved" dstcountry="Reserved" sessionid=1940841932 proto=17 action="deny" policyid=0 policytype="local-in-policy" service="SNM